In [ ]:
from conda_forge_tick.lazy_json_backends import LazyJson, lazy_json_override_backends

In [ ]:
import io
from collections.abc import Mapping, MutableSequence
from types import MethodType

from ruamel.yaml import YAML, CommentedMap

In [ ]:
# from https://stackoverflow.com/a/40227545
def _recursive_sort_data_by_keys(d):
    try:
        if isinstance(d, CommentedMap):
            return d.sort()
    except AttributeError:
        pass

    if isinstance(d, Mapping):
        # could use dict in newer python versions
        res = CommentedMap()
        for k in sorted(d.keys()):
            res[k] = _recursive_sort_data_by_keys(d[k])
        return res
    if isinstance(d, MutableSequence):
        for idx, elem in enumerate(d):
            d[idx] = _recursive_sort_data_by_keys(elem)
    return d


def get_yaml_parser(typ="rt", sort_keys=False):
    """Get a yaml parser.

    Parameters
    ----------
    typ : str
        The type of parser (e.g., 'rt', 'safe', 'jinja2').
    sort_keys : bool
        If True, sort keys on output.

    Returns
    -------
    parser
        A `ruamel.yaml.YAML` instance.
    """
    parser = YAML(typ=typ)  # spellchecker:disable-line
    parser.indent(mapping=2, sequence=4, offset=2)
    parser.width = 320
    parser.preserve_quotes = True
    parser.default_flow_style = False
    # do not use yaml anchors
    parser.representer.ignore_aliases = lambda x: True

    if sort_keys:
        orig_dump = parser.dump

        def dump(self, *args, **kwargs):
            args = list(args)
            args[0] = _recursive_sort_data_by_keys(args[0])
            args = tuple(args)
            orig_dump(*args, **kwargs)

        parser.dump = MethodType(dump, parser)

    def dumps(self, data):
        s = io.StringIO()
        self.dump(data, s)
        return s.getvalue()

    parser.dumps = MethodType(dumps, parser)
    parser.loads = parser.load

    return parser

In [ ]:
for bl in [True, False]:
    yaml = get_yaml_parser(sort_keys=bl)
    val = yaml.dumps({"c": 1, "blah": [None], "a": None})
    print(val)
    print(yaml.loads(val))
    print(yaml.loads("null: a"), yaml.loads('"": a'))

In [ ]:
import yaml

val = yaml.dump({None: 1, "blah": [None], "a": None})
print(val)
print(yaml.load(val, yaml.SafeLoader))
print(yaml.load("null: a", yaml.SafeLoader))
print(yaml.load('"": a', yaml.SafeLoader))

In [ ]:
yaml.representer

In [ ]:
!rm -rf node_attrs pr_info pr_json version_pr_info

In [ ]:
with lazy_json_override_backends(["github"], use_file_cache=True):
    attrs = LazyJson("node_attrs/ngmix.json")
    print(len(attrs.data))

In [ ]:
from collections.abc import Collection


def _sync_node(data, seen=None):
    seen = seen or []

    if isinstance(data, LazyJson):
        data.data

    if isinstance(data, Mapping):
        for v in data.values():
            if v not in seen:
                seen.append(v)
                seen = _sync_node(v, seen=seen)
    elif (
        isinstance(data, Collection)
        and not isinstance(data, str)
        and not isinstance(data, bytes)
    ):
        for v in data:
            if v not in seen:
                seen.append(v)
                seen = _sync_node(v, seen=seen)

    return seen


with lazy_json_override_backends(["github"]):
    ngmix = LazyJson("node_attrs/ngmix.json")
    _sync_node(ngmix)
    ngmix2 = LazyJson("node_attrs/ngmix.json")
    _sync_node(ngmix2)

    print(ngmix == ngmix2)

    with ngmix["pr_info"] as pri:
        pri.clear()
    print(ngmix == ngmix2)

    del ngmix.data["pr_info"]
    print(ngmix == ngmix2)

In [ ]:
import hashlib


def _get_names_for_job(names, job, n_jobs):
    job_index = job - 1
    return [
        node_id
        for node_id in names
        if abs(int(hashlib.sha1(node_id.encode("utf-8")).hexdigest(), 16)) % n_jobs
        == job_index
    ]


print(_get_names_for_job(["devtools"], 3, 3))

In [ ]:
s = "\tblah"

In [ ]:
print(s)

In [ ]:
s.startswith("\t")

In [ ]:
import secrets
import time

RNG = secrets.SystemRandom()


def _retry_sequence(num_tries=20, base=2, factor=0.01):
    for i in range(num_tries):
        start = factor * base**i
        end = factor * base ** (i + 1)
        time.sleep(RNG.uniform(start, end))
        yield i


for tr in _retry_sequence():
    print(tr)